## Data Preprocessing

This notebook prepares the dataset for analysis by:
- Cleaning and validating data
- Creating time-based and environmental features
- Defining the target variable (high severity)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Processed path:", DATA_PROCESSED)

Project root: D:\Data Analysis\Project  - Accidents
Processed path: D:\Data Analysis\Project  - Accidents\data\processed


### Load Processed Sample Data

Loading the sampled dataset created in the previous step.


In [3]:
file_path = DATA_PROCESSED / "accidents_sample_300k.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Shape:", df.shape)
df.head()

Shape: (300000, 8)


,Severity,Start_Time,Start_Lat,Start_Lng,Visibility(mi),Weather_Condition,Junction,Traffic_Signal
0,1,2020-04-17 09:29:30,26.706900,-80.119360,10.0,Mostly Cloudy,False,True
1,2,2022-04-21 10:01:00.000000000,38.781024,-121.265820,10.0,Mostly Cloudy,False,False
2,3,2016-08-12 16:45:00,33.985249,-84.269348,10.0,Partly Cloudy,False,False
3,3,2019-09-20 15:22:16,47.118706,-122.556908,10.0,Cloudy,False,False
4,2,2019-06-03 16:55:43,33.451355,-111.890343,10.0,Fair,False,False


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Severity           300000 non-null  int64  
 1   Start_Time         300000 non-null  str    
 2   Start_Lat          300000 non-null  float64
 3   Start_Lng          300000 non-null  float64
 4   Visibility(mi)     293202 non-null  float64
 5   Weather_Condition  293355 non-null  str    
 6   Junction           300000 non-null  bool   
 7   Traffic_Signal     300000 non-null  bool   
dtypes: bool(2), float64(3), int64(1), str(2)
memory usage: 14.3 MB


In [5]:
df.isnull().sum().sort_values(ascending=False)

Visibility(mi)       6798
Weather_Condition    6645
Severity                0
Start_Time              0
Start_Lng               0
Start_Lat               0
Junction                0
Traffic_Signal          0
dtype: int64

### Datetime Conversion

Converting Start_Time into datetime format for time-based feature extraction.

In [6]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")

In [7]:
print("Missing Start_Time after conversion:", df["Start_Time"].isna().sum())

Missing Start_Time after conversion: 28791


### Time Feature Extraction

Extracting year, month, hour, and day of week from Start_Time.

In [8]:
df["year"] = df["Start_Time"].dt.year
df["month"] = df["Start_Time"].dt.month
df["hour"] = df["Start_Time"].dt.hour
df["day_of_week"] = df["Start_Time"].dt.dayofweek

In [9]:
df[["Start_Time", "year", "month", "hour", "day_of_week"]].head()

,Start_Time,year,month,hour,day_of_week
0,2020-04-17 09:29:30,2020.0,4.0,9.0,4.0
1,NaT,NaN,NaN,NaN,NaN
2,2016-08-12 16:45:00,2016.0,8.0,16.0,4.0
3,2019-09-20 15:22:16,2019.0,9.0,15.0,4.0
4,2019-06-03 16:55:43,2019.0,6.0,16.0,0.0


### Year Filtering

Restricting data to the analysis period (2019–2023).

In [10]:
df = df[(df["year"] >= 2019) & (df["year"] <= 2023)].copy()

print("Shape after year filter:", df.shape)
print(df["year"].value_counts().sort_index())

Shape after year filter: (192487, 12)
year
2019.0    37111
2020.0    45131
2021.0    54938
2022.0    49015
2023.0     6292
Name: count, dtype: int64


### Target Variable Creation

Defining high_severity as a binary variable:
- 0 → low severity (1–2)
- 1 → high severity (3–4)

In [11]:
df["high_severity"] = np.where(df["Severity"] >= 3, 1, 0)

In [12]:
df["high_severity"].value_counts()

high_severity
0    162456
1     30031
Name: count, dtype: int64

In [13]:
df["high_severity"].value_counts(normalize=True)

high_severity
0    0.843984
1    0.156016
Name: proportion, dtype: float64

### Derived Time Features

Creating indicators for:
- Night conditions
- Peak traffic hours

In [14]:
df["night"] = np.where((df["hour"] >= 20) | (df["hour"] <= 5), 1, 0)

In [15]:
df["peak_hour"] = np.where(
    ((df["hour"] >= 7) & (df["hour"] <= 9)) |
    ((df["hour"] >= 16) & (df["hour"] <= 19)),
    1,
    0
)

### Weather-Based Features

Creating:
- adverse_weather indicator
- grouped weather categories

In [16]:
df["adverse_weather"] = df["Weather_Condition"].str.contains(
    "Rain|Snow|Fog|Storm|Hail",
    case=False,
    na=False
).astype(int)

In [17]:
def classify_weather(value):
    if pd.isna(value):
        return "Unknown"
    
    value = value.lower()
    
    if "clear" in value or "fair" in value:
        return "Clear/Fair"
    elif "cloud" in value or "overcast" in value:
        return "Cloudy"
    elif "rain" in value or "drizzle" in value:
        return "Rain"
    elif "snow" in value or "sleet" in value or "ice" in value:
        return "Snow/Ice"
    elif "fog" in value or "mist" in value or "haze" in value:
        return "Fog/Mist/Haze"
    elif "storm" in value or "thunder" in value:
        return "Storm"
    else:
        return "Other"

df["weather_group"] = df["Weather_Condition"].apply(classify_weather)

In [18]:
df["weather_group"].value_counts(dropna=False)

weather_group
Clear/Fair       89191
Cloudy           73551
Rain             13437
Fog/Mist/Haze     5118
Snow/Ice          4234
Unknown           4131
Storm             1934
Other              891
Name: count, dtype: int64

### Missing Value Handling

Handling missing values in key variables such as visibility and weather.


In [19]:
df["Visibility(mi)"] = df["Visibility(mi)"].fillna(df["Visibility(mi)"].median())

In [20]:
df["Visibility(mi)"].isna().sum()

np.int64(0)

In [21]:
df.info()

<class 'pandas.DataFrame'>
Index: 192487 entries, 0 to 299999
Data columns (total 17 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Severity           192487 non-null  int64         
 1   Start_Time         192487 non-null  datetime64[us]
 2   Start_Lat          192487 non-null  float64       
 3   Start_Lng          192487 non-null  float64       
 4   Visibility(mi)     192487 non-null  float64       
 5   Weather_Condition  188356 non-null  str           
 6   Junction           192487 non-null  bool          
 7   Traffic_Signal     192487 non-null  bool          
 8   year               192487 non-null  float64       
 9   month              192487 non-null  float64       
 10  hour               192487 non-null  float64       
 11  day_of_week        192487 non-null  float64       
 12  high_severity      192487 non-null  int64         
 13  night              192487 non-null  int64         
 14  peak

In [22]:
df[[
    "Severity", "high_severity", "Start_Time", "year", "month", "hour",
    "day_of_week", "night", "peak_hour", "Weather_Condition",
    "weather_group", "adverse_weather", "Visibility(mi)",
    "Junction", "Traffic_Signal"
]].head()

,Severity,high_severity,Start_Time,year,month,hour,day_of_week,night,peak_hour,Weather_Condition,weather_group,adverse_weather,Visibility(mi),Junction,Traffic_Signal
0,1,0,2020-04-17 09:29:30,2020.0,4.0,9.0,4.0,0,1,Mostly Cloudy,Cloudy,0,10.0,False,True
3,3,1,2019-09-20 15:22:16,2019.0,9.0,15.0,4.0,0,0,Cloudy,Cloudy,0,10.0,False,False
4,2,0,2019-06-03 16:55:43,2019.0,6.0,16.0,0.0,0,1,Fair,Clear/Fair,0,10.0,False,False
5,2,0,2021-02-04 12:48:21,2021.0,2.0,12.0,3.0,0,0,Light Snow / Windy,Snow/Ice,1,0.5,False,False
6,2,0,2022-06-23 10:57:30,2022.0,6.0,10.0,3.0,0,0,Cloudy,Cloudy,0,10.0,False,False


### Save Cleaned Dataset

Saving the processed dataset for downstream analysis and modeling.


In [23]:
clean_path = DATA_PROCESSED / "accidents_cleaned.csv"
df.to_csv(clean_path, index=False)

print("Saved cleaned dataset to:", clean_path)
print("File exists:", clean_path.exists())

Saved cleaned dataset to: D:\Data Analysis\Project  - Accidents\data\processed\accidents_cleaned.csv
File exists: True
